# Phase 2 - Try Resizing a Running Job

Two tests:

1. **Through Kubeflow Trainer** - start a TrainJob, try to patch `numNodes`.
   Expect it to be rejected. The error message is the deliverable.

2. **Through JobSet directly** - enable `ElasticJobSet` feature gate, start a
   job on JobSet, change parallelism up and down. If JobSet works but Trainer
   doesn't, that proves the gap is in the Trainer layer.

## Setup

In [1]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"
!pip install yamlmagic hatchling --index-url https://pypi.org/simple
!pip install --no-deps .. --index-url https://pypi.org/simple
%load_ext yamlmagic

Looking in indexes: https://console.redhat.com/api/pypi/public-rhai/rhoai/3.4/cuda13.0-ubi9/simple/
  Cloning https://github.com/opendatahub-io/kubeflow-sdk.git (to revision v0.3.0+rhaiv.2) to /tmp/pip-install-1q52dud5/kubeflow_027146179ce34649a227b9276ef4ae4a
  Running command git clone --filter=blob:none --quiet https://github.com/opendatahub-io/kubeflow-sdk.git /tmp/pip-install-1q52dud5/kubeflow_027146179ce34649a227b9276ef4ae4a
  Running command git checkout -q 3fa50fe903f0fedc6429485a6c6c7105bc9149bc
  Resolved https://github.com/opendatahub-io/kubeflow-sdk.git to commit 3fa50fe903f0fedc6429485a6c6c7105bc9149bc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.0/644.0 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.4/740.4 kB 49.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 152.4 MB/s  0:00:00
   ━━━━━━━━

In [3]:
%%yaml parameters

namespace: elastic-scaling
duration_minutes: 10

<IPython.core.display.Javascript object>

In [4]:
%load_ext autoreload
%autoreload 2

from elastic_scaling_poc.phase2.resize_test import train_func

print("train_func loaded")

train_func loaded


In [5]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

api_server = os.environ["OPENSHIFT_API_URL"]
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not token:
    sa_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_path.exists():
        token = sa_path.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench "
        "with a service-account token."
    )

config = k8s.Configuration()
config.host = api_server
config.api_key = {"authorization": f"Bearer {token}"}
config.verify_ssl = False

client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=config,
    )
)

/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [6]:
!oc login {config.host} --token={token} --insecure-skip-tls-verify=true


Logged into "https://api.dmytro-test-pool-998r5.aws.rh-ods.com:6443" as "htpasswd-cluster-admin-user" using the token provided.

You have access to 90 projects, the list has been suppressed. You can list all projects with 'oc projects'

Using project "default".


## Part 1 - Resize through Kubeflow Trainer

Submit a TrainJob on 1 GPU, wait for it to start, then try to patch
`numNodes` from 1 to 2. The research doc predicts two blockers:

1. `TrainJob.spec.trainer` is immutable - the patch is refused before
   it even reaches JobSet.
2. Even if it did reach JobSet, the `PET_NNODES` env var change would
   trigger the webhook rejection.

The error message itself is the deliverable for this part.

In [7]:
from kubeflow.trainer.options import Name
from kubeflow.trainer.rhai import TransformersTrainer

trainer = TransformersTrainer(
    func=train_func,
    func_args=parameters,
    num_nodes=1,
    resources_per_node={"nvidia.com/gpu": 1},
)

runtime = client.backend.get_runtime("torch-distributed")
JOB_NAME = client.train(
    trainer=trainer,
    runtime=runtime,
    options=[Name("phase2-resize-trainer")],
)
print(f"Submitted: {JOB_NAME}")

Submitted: phase2-resize-trainer


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-

In [8]:
import time

# Wait for the job to start running
for _ in range(60):
    status = client.get_job(JOB_NAME).status
    print(f"Status: {status}")
    if status == "Running":
        break
    time.sleep(10)

Status: Running


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-

### Try to patch numNodes

This should fail. Capture the exact error.

In [9]:
!oc patch trainjob phase2-resize-trainer -n {parameters["namespace"]} \
    --type=merge -p '{{"spec": {{"trainer": {{"numNodes": 2}}}}}}'

trainjob.trainer.kubeflow.org/phase2-resize-trainer patched


### Verify the patch

Check if `numNodes` changed on the TrainJob, and whether it propagated to the underlying JobSet and pods.

In [10]:
# TrainJob spec after patch
!oc get trainjob phase2-resize-trainer -n {parameters["namespace"]} \
    -o jsonpath='numNodes: {{.spec.trainer.numNodes}}'

numNodes: 2

In [14]:
# Underlying JobSet - did parallelism change?
!oc get jobset phase2-resize-trainer -n {parameters["namespace"]} \
    -o jsonpath='parallelism: {{.spec.replicatedJobs[0].template.spec.parallelism}}  completions: {{.spec.replicatedJobs[0].template.spec.completions}}'

parallelism: 1  completions: 1

In [16]:
# Pods - did a second worker appear?
!oc get pods -n {parameters["namespace"]} -l jobset.sigs.k8s.io/jobset-name=phase2-resize-trainer

NAME                                   READY   STATUS    RESTARTS   AGE
phase2-resize-trainer-node-0-0-hd7tg   1/1     Running   0          3m6s


In [17]:
# Cleanup Part 1
client.delete_job(name=JOB_NAME)
print(f"Deleted {JOB_NAME}")

Deleted phase2-resize-trainer


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.dmytro-test-pool-998r5.aws.rh-ods.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Part 2 - Resize through JobSet directly

Skip Kubeflow Trainer entirely. Create a JobSet with `completionMode: Indexed`
and the `ElasticJobSet` feature gate enabled on the controller.

If this works but Part 1 doesn't, it proves the gap is specifically in the
Trainer layer (immutable spec + PET_NNODES coupling).

**Prerequisite:** ElasticJobSet feature gate must be enabled on the JobSet
controller. See the research doc section 8 for how to do this.

In [18]:
# Verify the feature gate is active
!oc get configmap jobset-manager-config -n openshift-jobset-operator \
    -o jsonpath='{{.data.controller_manager_config\.yaml}}' | grep -A1 featureGates

featureGates:
  ElasticJobSet: true


In [19]:
!oc apply -f ../src/elastic_scaling_poc/phase2/manifests/jobset-resize-test.yaml

jobset.jobset.x-k8s.io/phase2-resize-jobset created


In [20]:
# Wait for it to start
import time

for _ in range(60):
    result = !oc get jobset phase2-resize-jobset -n {parameters["namespace"]} -o jsonpath='{{.status.replicatedJobsStatus[0].ready}}'
    ready = result[0] if result else "0"
    print(f"Ready pods: {ready}")
    if ready == "1":
        break
    time.sleep(10)

Ready pods: 0
Ready pods: 1


### Try to scale up: parallelism 1 -> 2

In [21]:
!oc patch jobset phase2-resize-jobset -n {parameters["namespace"]} \
    --type=json -p '[{{"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/parallelism", "value": 2}}, {{"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/completions", "value": 2}}]'

jobset.jobset.x-k8s.io/phase2-resize-jobset patched


In [22]:
# Pods after scale-up
!oc get pods -n {parameters["namespace"]} -l jobset.sigs.k8s.io/jobset-name=phase2-resize-jobset

NAME                                     READY   STATUS    RESTARTS   AGE
phase2-resize-jobset-workers-0-0-f5kb2   1/1     Running   0          31s
phase2-resize-jobset-workers-0-1-sm49j   1/1     Running   0          8s


In [23]:
# JobSet spec after scale-up
!oc get jobset phase2-resize-jobset -n {parameters["namespace"]} \
    -o jsonpath='parallelism: {{.spec.replicatedJobs[0].template.spec.parallelism}}  completions: {{.spec.replicatedJobs[0].template.spec.completions}}'

parallelism: 2  completions: 2

### Try to scale down: parallelism 2 -> 1

In [24]:
!oc patch jobset phase2-resize-jobset -n {parameters["namespace"]} \
    --type=json -p '[{{"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/parallelism", "value": 1}}, {{"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/completions", "value": 1}}]'

jobset.jobset.x-k8s.io/phase2-resize-jobset patched


In [25]:
# Pods after scale-down
!oc get pods -n {parameters["namespace"]} -l jobset.sigs.k8s.io/jobset-name=phase2-resize-jobset

NAME                                     READY   STATUS        RESTARTS   AGE
phase2-resize-jobset-workers-0-0-f5kb2   1/1     Running       0          51s
phase2-resize-jobset-workers-0-1-sm49j   1/1     Terminating   0          28s


In [26]:
# JobSet spec after scale-down
!oc get jobset phase2-resize-jobset -n {parameters["namespace"]} \
    -o jsonpath='parallelism: {{.spec.replicatedJobs[0].template.spec.parallelism}}  completions: {{.spec.replicatedJobs[0].template.spec.completions}}'

parallelism: 1  completions: 1

## Cleanup

In [27]:
!oc delete jobset phase2-resize-jobset -n {parameters["namespace"]} --ignore-not-found
!oc delete trainjob phase2-resize-trainer -n {parameters["namespace"]} --ignore-not-found

jobset.jobset.x-k8s.io "phase2-resize-jobset" deleted from elastic-scaling namespace


## Results

Record what happened:

| Test | Expected | Actual | Error message |
|------|----------|--------|---------------|
| Patch TrainJob numNodes | Rejected (immutable spec) | | |
| Scale up JobSet parallelism | Accepted (ElasticJobSet) | | |
| Scale down JobSet parallelism | Accepted (ElasticJobSet) | | |

If Part 1 fails and Part 2 succeeds, the gap is confirmed: the blocker is
in the Kubeflow Trainer layer, not in JobSet.